# 05 — Inference Demo

This notebook loads the saved model and predicts emotions for custom text.

Use this notebook to test whether the trained model works before connecting it to a web app, chatbot, or backend.

In [ ]:
# Import required libraries
from pathlib import Path
import json

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

## 1. Set paths and load model

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_DIR = PROJECT_ROOT / "models" / "emotion_model_v2"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
print("Model directory:", MODEL_DIR)

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForSequenceClassification.from_pretrained(str(MODEL_DIR))
model.to(device)
model.eval()

print("Model loaded successfully.")

## 2. Create prediction function

In [ ]:
def predict_emotion(text, return_scores=True):
    # Predict emotion from text.
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=-1)[0]

    predicted_id = int(torch.argmax(probabilities).item())
    predicted_label = model.config.id2label[predicted_id]
    confidence = float(probabilities[predicted_id].item())

    result = {
        "text": text,
        "predicted_emotion": predicted_label,
        "confidence": round(confidence, 4)
    }

    if return_scores:
        all_scores = {
            model.config.id2label[i]: round(float(probabilities[i].item()), 4)
            for i in range(len(probabilities))
        }
        result["all_scores"] = all_scores

    return result

## 3. Test with sample sentences

In [ ]:
sample_texts = [
    "I am so happy and excited today!",
    "Thank you so much, I really appreciate your help.",
    "I feel very sad and disappointed.",
    "I am angry because this is not fair.",
    "I am scared about what will happen next.",
    "I feel nervous and stressed about my exam.",
    "I do not understand what is going on here.",
    "Okay, that sounds fine to me."
]

results = []

for text in sample_texts:
    prediction = predict_emotion(text, return_scores=False)
    results.append(prediction)

results_df = pd.DataFrame(results)
display(results_df)

## 4. Check full probability scores for one sentence

In [ ]:
text = "I am worried about my results and I cannot relax."

prediction = predict_emotion(text, return_scores=True)

print("Text:", prediction["text"])
print("Predicted emotion:", prediction["predicted_emotion"])
print("Confidence:", prediction["confidence"])

display(pd.DataFrame(
    prediction["all_scores"].items(),
    columns=["emotion", "probability"]
).sort_values("probability", ascending=False))

## 5. Try your own sentence

In [ ]:
# Change this sentence and run the cell again.
my_text = "I am confused about this project but I want to learn."

predict_emotion(my_text, return_scores=True)

## Inference Summary

This notebook confirms whether the saved model can classify new text into the 7 project emotions.

For a final app, you can convert the prediction function into a small Python file such as:

```text
app/classifier.py
```